In [1]:
# Standard library imports
import sys, os, time

work_dir = "/home/yinchang/DeepQMC"
os.chdir(work_dir)


# Third-party library imports
import jax
import kfac_jax
import numpy as np
import ml_collections
from jax import numpy as jnp
from absl import logging


# Local module imports
from src.utils import utils, system
from src import checkpoint, pretrain, constants
from src.trainer import train, inference
from src.initialization import initializer
from src.config import base_config


############################################################ 
#                                                          #
#                   Configuration Setting                  #
#                                                          #
############################################################
cfg = base_config.default()
# Set up molecule spins, atomic types and positions
cfg.system.electrons = (2, 2)
atoms_angstrom = [
    ('H', [0.0, 0.0, 0.0]),
    ('H', [1.0, 0.0, 0.0]), 
    ('H', [2.0, 0.0, 0.0]), 
    ('H', [3.0, 0.0, 0.0])
    ]

ang_to_bohr = 1.0 / 0.529177

cfg.system.molecule = []
for symbol, coords_ang in atoms_angstrom:
    coords_bohr = [coord * ang_to_bohr for coord in coords_ang]
    cfg.system.molecule.append(system.Atom(symbol, coords_bohr))

# Debug mode settngs
cfg.debug.deterministic = False

# Mode
cfg.mode = 'training'

# Network settings
cfg.batch_size = 256
cfg.network.psiformer.use_edge_bias = True #####
cfg.network.rescale_inputs = True
cfg.network.psiformer.heads_dim = 64
cfg.network.psiformer.num_heads = 4
cfg.network.jastrow = 'exp' #####

# MCMC settings
cfg.mcmc.proposal = 'random_walk'
cfg.mcmc.move_width = 0.02
cfg.mcmc.burn_in = 100
cfg.mcmc.adapt_frequency = 100
#cfg.mcmc.max_norm = 5.0

# Training settings
cfg.optim.optimizer = 'kfac'
cfg.pretrain.iterations = 100
cfg.optim.iterations = 500
#cfg.optim.reg_weight = 1e-3  # use entropy regularization

# Log settings
#cfg.log.show_jastrow_factor_params = True

# Checkpoint settings
cfg.log.max_to_keep = 10
cfg.log.save_interval_steps = 500
cfg.log.restore_path = ""#"/home/yinchang/ChemFM/checkpoints/test_1"
cfg.log.save_path = "/home/yinchang/ChemFM/checkpoints"

In [2]:
# Third-party library imports
import jax
import kfac_jax
import numpy as np
import ml_collections
from jax import numpy as jnp
from absl import logging


# Local module imports
from src.utils import utils
from src import checkpoint, pretrain, constants
from src.trainer import train, inference
from src.initialization import initializer
from src.modules import radial



logging.set_verbosity(logging.INFO)  # Logging level
logging.info(cfg)
############################################################ 
#                                                          #
#                      Initialization                      #
#                                                          #
############################################################
# Random seed and key
if cfg.debug.deterministic:
    seed = 23
    logging.info(f"DEBUG mode enabled, using a fixed random number seed={seed}. "
                f"This will be overridden when checkpoint loaded.")
else:
    seed = int(time.time() * 1e6)
key = jax.random.PRNGKey(seed)

# Create QMCInitializer
init = initializer.QMCInitializer(cfg, key)
# Input data
data = init.init_data()

INFO:absl:batch_size: 256
config_module: src.config.base_config
debug:
  deterministic: false
log:
  max_to_keep: 10
  restore_path: ''
  save_interval_steps: 500
  save_path: /home/yinchang/ChemFM/checkpoints
  show_det_weights: false
  show_jastrow_factor_params: false
  show_params: false
  stats_frequency: 1
mcmc:
  adapt_frequency: 100
  burn_in: 100
  init_width: 1.0
  max_norm: 5.0
  move_width: 0.02
  proposal: random_walk
  sampler_params: {}
  steps: 30
mode: training
network:
  activation_fun: tanh
  bias_orbitals: true
  determinants: 16
  envelope: simple
  full_det: true
  jastrow: exp
  network_type: psiformer
  psiformer:
    heads_dim: 64
    mlp_hidden_dims: 256
    num_heads: 4
    num_layers: 4
    separate_spin_channels: true
    use_edge_bias: true
    use_gate: false
    use_layer_norm: true
    use_res: true
  rescale_inputs: true
optim:
  adam:
    b1: 0.9
    b2: 0.999
    eps: 1.0e-08
    eps_root: 0.0
  center_at_clip: true
  clip_local_energy: 5.0
  clip_me

In [3]:
from src.networks import ChemFM

from src.modules import network_blocks

model = ChemFM.NucleiMPNN(
    max_species = 10, 
    num_layers = 1,
    num_mlp_layers = 2,
    dim_mlp = 64,
    max_ell = 3, 
    num_output_irreps = 64,
    radial_basis = radial.default_radial_basis,
    n_radial_basis = 8,
    act_fn = jax.nn.tanh,
    ndim = cfg.system.ndim
    )

key, subkey = jax.random.split(key)
params = model.init(
        subkey, 
        data.atoms[0][0], 
        data.charges[0][0]
        )
del subkey

In [4]:
def get_param_size(params):
        param_sizes = jax.tree_util.tree_map(
            lambda x: x.size, params['params'])
        num_params = sum(jax.tree_util.tree_leaves(param_sizes))
        
        return num_params

In [5]:
print(f"Number of parameters: {get_param_size(params)}")

Number of parameters: 169152


In [6]:
na = 4
senders = []
receivers = []

for i in range(na):
    for j in range(na):
        if i != j:
            senders.append(i)
            receivers.append(j)

print(senders)
print(receivers)

[0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3]
[1, 2, 3, 0, 2, 3, 0, 1, 3, 0, 1, 2]


In [7]:
output = model.apply(params, data.atoms[0][0], data.charges[0][0])

In [8]:
print(output[0])
print(output[1])
print(output[2])
print(output[3])
print(output[4])

64x0e+64x1o+64x2e
[[-0.02499766  0.1969718   0.45460805 ...  0.          0.
   0.        ]
 [-0.02499766  0.1969718   0.45460805 ...  0.          0.
   0.        ]
 [-0.02499766  0.1969718   0.45460805 ...  0.          0.
   0.        ]
 [-0.02499766  0.1969718   0.45460805 ...  0.          0.
   0.        ]]
1x1o
[[ 1.8897269  0.         0.       ]
 [ 3.7794538  0.         0.       ]
 [ 5.669181   0.         0.       ]
 [-1.8897269  0.         0.       ]
 [ 1.8897269  0.         0.       ]
 [ 3.779454   0.         0.       ]
 [-3.7794538  0.         0.       ]
 [-1.8897269  0.         0.       ]
 [ 1.8897271  0.         0.       ]
 [-5.669181   0.         0.       ]
 [-3.779454   0.         0.       ]
 [-1.8897271  0.         0.       ]]
[[1.8897269]
 [3.7794538]
 [5.669181 ]
 [1.8897269]
 [1.8897269]
 [3.779454 ]
 [3.7794538]
 [1.8897269]
 [1.8897271]
 [5.669181 ]
 [3.779454 ]
 [1.8897271]]
[0 0 0 1 1 1 2 2 2 3 3 3]
[1 2 3 0 2 3 0 1 3 0 1 2]


In [23]:
input = output[2][:, 0]
print(f"input:\n{input}")
output = radial.default_radial_basis(input, 8)
print(f"output:\n{output}")

input:
[1.8897269 3.7794538 5.669181  1.8897269 1.8897269 3.779454  3.7794538
 1.8897269 1.8897271 5.669181  3.779454  1.8897271]
output:
[[-0. -0. -0. -0. -0. -0. -0. -0.]
 [-0. -0. -0. -0.  0.  0.  0.  0.]
 [-0. -0. -0.  0.  0.  0. -0. -0.]
 [-0. -0. -0. -0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0. -0. -0. -0.]
 [-0. -0. -0. -0.  0.  0.  0.  0.]
 [-0. -0. -0. -0.  0.  0.  0.  0.]
 [-0. -0. -0. -0. -0. -0. -0. -0.]
 [-0. -0. -0. -0. -0. -0. -0. -0.]
 [-0. -0. -0.  0.  0.  0. -0. -0.]
 [-0. -0. -0. -0.  0.  0.  0.  0.]
 [-0. -0. -0. -0. -0. -0. -0. -0.]]


In [29]:
radial.default_radial_basis(jnp.array([0.5]), 8)

Array([[ 2.4196310e+00, -2.1153086e-07, -2.4196310e+00,  4.2306172e-07,
         2.4196310e+00, -5.7707620e-08, -2.4196310e+00,  8.4612344e-07]],      dtype=float32)

In [18]:
print(output[0][0])

64x0e+64x1o+64x2e
[-0.02499766  0.1969718   0.45460805 -0.02315109  0.12744126 -0.04037143
  0.3624174   0.20757563 -0.13930541  0.10842109  0.06583979 -0.30130517
 -0.12229475  0.39773887  0.25064382  0.70493597 -0.34566486  0.8090504
  0.41836384 -0.18290666  0.5762864  -0.29682168  0.29978427  0.3880574
  0.43651536  0.26711532  0.56651163  0.1732876   0.26119396 -0.03669398
  1.2973921   1.1023585   0.41447568  0.17626356 -0.08674898 -0.10012784
 -0.2836611  -0.26165748  0.14295648  0.05829537  0.13917959  0.3215135
  0.3059468  -0.07288426 -0.20976682  0.09385378 -0.07629868  0.09829187
 -0.08238595  0.21668576  0.66346025 -0.11284415 -0.30227312  0.20403361
 -0.05361916 -0.20090538 -0.28578702  0.35588053  0.39599052 -0.16683905
  0.40860784 -0.2740357   0.06528511  0.02614009  0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.        

In [10]:
from flax.nnx import display

display(params)